# Offline Policy Evaluation


### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

In [6]:
evaluator.evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 304.31it/s]


2026-06-08 04:42:09.231 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:903 - Data batch-empirical estimation of propensity score.


2026-06-08 04:42:09.238 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:952 - Data prediction of expected reward based on gbm model.


2026-06-08 04:42:10.643 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1069 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


...

2026-06-08 04:42:40.789 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-06-08 04:42:41.092 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-06-08 04:42:41.397 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.513776,0.480250,0.547661,0.017202,b-ipw,reward_0
1,0.505128,0.504910,0.505353,0.000113,dm,reward_0
2,0.510443,0.478852,0.543767,0.016579,dr,reward_0
3,0.505128,0.504910,0.505365,0.000115,dros-opt,reward_0
4,0.510443,0.479438,0.543361,0.016305,dros-pess,reward_0
5,0.510295,0.476592,0.543819,0.016967,ipw,reward_0
6,0.510039,0.478225,0.544452,0.016997,rep,reward_0
7,0.510444,0.477510,0.542061,0.016410,sndr,reward_0
8,0.510420,0.477747,0.543553,0.016833,snips,reward_0
9,0.510443,0.477727,0.542804,0.016615,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 329.21it/s]


2026-06-08 04:42:41.853 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1301 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


SVI:   0%|          | 0/1000 [00:00<?, ?it/s]

SVI:   0%|          | 1/1000 [00:00<07:55,  2.10it/s]

...

2026-06-08 04:43:29.597 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-06-08 04:43:29.903 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-06-08 04:43:30.208 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.523545,0.490816,0.557698,0.016985,b-ipw,reward_0
1,0.505119,0.504849,0.505392,0.000139,dm,reward_0
2,0.520437,0.485758,0.553743,0.017257,dr,reward_0
3,0.505119,0.504839,0.505396,0.000141,dros-opt,reward_0
4,0.520437,0.487400,0.554758,0.017166,dros-pess,reward_0
5,0.526642,0.490340,0.562833,0.018507,ipw,reward_0
6,0.520000,0.484797,0.555406,0.018100,rep,reward_0
7,0.520256,0.486177,0.553668,0.017122,sndr,reward_0
8,0.520404,0.485620,0.557272,0.018398,snips,reward_0
9,0.520437,0.486882,0.554090,0.017282,sg-dr,reward_0
